In [12]:
# ────────────── LOCALIZA LA RAÍZ DEL PROYECTO ──────────────
import sys, pathlib, json, joblib, pandas as pd, lightgbm as lgb

cur = pathlib.Path.cwd()
for p in [cur] + list(cur.parents):
    if (p / "memebot3").is_dir():
        PROJECT_ROOT = p            # carpeta que contiene /memebot3
        break
else:
    raise RuntimeError("No se encuentra la carpeta ‘memebot3’")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ─────────── IMPORTS DEL PROYECTO ───────────
from memebot3.features.builder import COLUMNS

# ─────────────── CARGA DEL PARQUET ──────────
PARQUET = PROJECT_ROOT / "memebot3" / "data" / "features" / "features_202507.parquet"
assert PARQUET.exists(), f"Parquet no encontrado: {PARQUET}"
df = pd.read_parquet(PARQUET)

# ⭑ CAMBIOS → dejamos sólo columnas numéricas / bool
obj_cols = df.select_dtypes(include=["object", "datetime"]).columns
X_cols   = [c for c in COLUMNS if c not in obj_cols]

# forzamos bool→int8  (cluster_bad, social_ok, …)
for col in X_cols:
    if df[col].dtype == "bool":
        df[col] = df[col].astype("int8")

X = df[X_cols].apply(pd.to_numeric, errors="coerce").fillna(0).astype("float32")
y = df["label"].astype("int8")

# ───────────── ENTRENAMIENTO ────────────────
split = int(len(df) * 0.8)
dtrain = lgb.Dataset(X.iloc[:split],  label=y.iloc[:split],  free_raw_data=False)
dvalid = lgb.Dataset(X.iloc[split:], label=y.iloc[split:], free_raw_data=False)

params = dict(
    objective="binary",
    metric="auc",
    learning_rate=0.05,
    num_leaves=64,
    min_data_in_leaf=100,
    subsample=0.8,
    colsample_bytree=0.8,
    seed=42,
    verbosity=-1,
)

model = lgb.train(
    params,
    dtrain,
    num_boost_round=800,
    valid_sets=[dtrain, dvalid],
    valid_names=["tr", "va"],
    callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)],
)

print("AUC valid =", model.best_score["va"]["auc"])

# ───────────── GUARDAR MODELO ───────────────
MODEL_DIR = PROJECT_ROOT / "memebot3" / "ml"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(model, MODEL_DIR / "model.pkl")
json.dump({"features": X_cols}, open(MODEL_DIR / "model.meta.json", "w"))

print("✔  model.pkl + model.meta.json guardados en", MODEL_DIR)


AUC valid = 1.0
✔  model.pkl + model.meta.json guardados en d:\Dev\Python\memebot3\ml
